# Virus H3 coverage maps

Interactive Folium maps of **H3 resolution-4** cells occupied by virus detections in `data/02_interim/viruses_normalized.parquet`.

**What hexes mean:** a cell is drawn if the extract has ≥1 dated detection with an assigned H3 cell there (after optional filters). Only rows with non-null `h3_cell`, `year`, and `month` are mapped — country-only / undated records stay off the map. Empty space is *not* proven pathogen absence.

**Metrics are stored per `(h3_cell, pathogen_name)`:**
- `n_year_months` — how often detections appear (distinct year×month)
- `n_years` / `year_span` — how long the record spans
- `n_records` — raw detection row count (effort-sensitive)

**Map UX:** one polygon per H3 cell on an OpenStreetMap street basemap (hex fill is semi-transparent). Fill color uses the **max** of the plotted metric across pathogens in that cell (shared color scale).

**Hover or click** a hex for a **horizontal** table (metric rows × one column per pathogen). When several pathogens are present, the **layer control** (top-right) toggles pathogens only (H3 polygons stay always on): unchecked pathogens disappear from the popup, and hexes that have **no remaining active pathogen** are hidden.

Set `PATHOGEN_NAME` to filter to one pathogen, or `None` for all pathogens.

After pulling code changes: **restart the kernel** and re-run from the top so `vector_guard.viz` reloads.

Edit the **config** cell (`COUNTRY_ISO2`, `PATHOGEN_NAME`) and re-run from there to redraw.

In [1]:
from pathlib import Path

import pandas as pd

from vector_guard.data import default_interim_dir, find_repo_root
from vector_guard.viz import aggregate_interim_h3_cells, folium_h3_choropleth

REPO_ROOT = find_repo_root(Path.cwd())
VIRUS_PATH = default_interim_dir(REPO_ROOT) / "viruses_normalized.parquet"
viruses = pd.read_parquet(VIRUS_PATH)
print(f"Loaded {len(viruses):,} rows from {VIRUS_PATH.relative_to(REPO_ROOT)}")
print("With H3:", int(viruses["h3_cell"].notna().sum()), f"/ {len(viruses):,}")
print("Countries:", viruses["country_iso2"].nunique())
print("Pathogens:", sorted(viruses["pathogen_name"].dropna().unique()))

Loaded 168,307 rows from data/02_interim/viruses_normalized.parquet
With H3: 77485 / 168,307
Countries: 220
Pathogens: ['African horse sickness virus', 'Akabane orthobunyavirus', 'Alongshan virus', 'Batai orthobunyavirus', 'Bluetongue virus', 'Chikungunya virus', 'Crimean-Congo hemorrhagic fever virus', 'Dengue virus', 'Epizootic hemorrhagic disease virus', 'Inkoo orthobunyavirus', 'Japanese encephalitis virus', 'Jingmen tick virus', 'Louping ill virus', 'Omsk hemorrhagic fever virus', 'Phlebovirus (Mediterranean sandfly fever complex)', 'Rift Valley fever virus', 'Sandfly fever Naples phlebovirus / Toscana virus', 'Schmallenberg virus', 'Sindbis virus', 'Tahyna / California encephalitis orthobunyavirus', 'Thogoto / Bourbon thogotovirus', 'Tick-borne encephalitis virus', 'Usutu virus', 'West Nile virus', 'Yellow fever virus', 'Zika virus']


## Config

Set `COUNTRY_ISO2` to an ISO-2 code (e.g. `"ES"`) to filter, or `None` for all countries.  
Set `PATHOGEN_NAME` to a pathogen string (e.g. `"West Nile virus"`) for a single pathogen, or `None` to include all pathogens (multi-pathogen cells list every pathogen in the click popup).

In [2]:
COUNTRY_ISO2 = None  # e.g. "ES"; None = all countries
PATHOGEN_NAME = None  # e.g. "West Nile virus"; None = all pathogens

In [3]:
filtered = viruses.copy()
if COUNTRY_ISO2 is not None:
    code = str(COUNTRY_ISO2).strip().upper()
    filtered = filtered.loc[filtered["country_iso2"].astype(str).str.upper() == code]
if PATHOGEN_NAME is not None:
    name = str(PATHOGEN_NAME).strip()
    filtered = filtered.loc[filtered["pathogen_name"].astype(str) == name]

cell_stats = aggregate_interim_h3_cells(
    viruses,
    group_col="pathogen_name",
    country_iso2=COUNTRY_ISO2,
    group_value=PATHOGEN_NAME,
)

n_cells = int(cell_stats["h3_cell"].nunique()) if not cell_stats.empty else 0
n_multi = (
    int((cell_stats.groupby("h3_cell").size() >= 2).sum()) if not cell_stats.empty else 0
)

print(
    f"Filter: country={COUNTRY_ISO2!r}, pathogen_name={PATHOGEN_NAME!r}"
)
print(f"Rows after filter: {len(filtered):,} / {len(viruses):,}")
print(f"(h3_cell, pathogen_name) units: {len(cell_stats):,}")
print(f"Unique H3 cells: {n_cells:,}")
print(f"H3 cells with ≥2 pathogens: {n_multi:,}")
if not cell_stats.empty:
    print("Units by pathogen_name:")
    display(cell_stats["pathogen_name"].value_counts())
    display(
        cell_stats[["n_year_months", "n_years", "year_span", "n_records"]].describe()
    )

Filter: country=None, pathogen_name=None
Rows after filter: 168,307 / 168,307
(h3_cell, pathogen_name) units: 4,722
Unique H3 cells: 3,510
H3 cells with ≥2 pathogens: 934
Units by pathogen_name:


pathogen_name
Bluetongue virus                                     1509
West Nile virus                                       748
Rift Valley fever virus                               645
Schmallenberg virus                                   452
Dengue virus                                          374
Chikungunya virus                                     214
Japanese encephalitis virus                           138
Zika virus                                            119
Usutu virus                                           114
African horse sickness virus                          112
Yellow fever virus                                     82
Crimean-Congo hemorrhagic fever virus                  43
Epizootic hemorrhagic disease virus                    39
Tick-borne encephalitis virus                          38
Alongshan virus                                        23
Jingmen tick virus                                     21
Sandfly fever Naples phlebovirus / Toscana virus       16


,n_year_months,n_years,year_span,n_records
count,4722.000000,4722.000000,4722.000000,4722.000000
mean,2.989411,1.675561,3.128547,13.613511
std,4.669720,1.567851,5.103441,52.942867
min,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000
50%,2.000000,1.000000,1.000000,3.000000
75%,3.000000,2.000000,2.000000,9.000000
max,96.000000,21.000000,88.000000,1272.000000


## Frequency map (`n_year_months`)

Fill color = **max** `n_year_months` across pathogens in the cell. Hover/click a hex for a horizontal stats table (one column per pathogen).

In [4]:
freq_title = "n_year_months"
if COUNTRY_ISO2:
    freq_title += f" · {str(COUNTRY_ISO2).upper()}"
if PATHOGEN_NAME:
    freq_title += f" · {PATHOGEN_NAME}"

folium_h3_choropleth(
    cell_stats,
    "n_year_months",
    group_col="pathogen_name",
    title=freq_title,
)

## Duration map (`n_years`)

Fill color = **max** `n_years` across pathogens in the cell. Hover/click a hex for a horizontal stats table (one column per pathogen).

In [5]:
dur_title = "n_years"
if COUNTRY_ISO2:
    dur_title += f" · {str(COUNTRY_ISO2).upper()}"
if PATHOGEN_NAME:
    dur_title += f" · {PATHOGEN_NAME}"

folium_h3_choropleth(
    cell_stats,
    "n_years",
    group_col="pathogen_name",
    title=dur_title,
)

## Top cells

Highest frequency and longest duration under the current filter (one row per `h3_cell` × `pathogen_name`).

In [6]:
cols = [
    "h3_cell",
    "pathogen_name",
    "n_year_months",
    "n_years",
    "year_span",
    "n_records",
    "year_min",
    "year_max",
]

print("Top by n_year_months")
display(cell_stats[cols].head(15))

print("Top by n_years")
display(
    cell_stats.sort_values(
        ["n_years", "year_span", "n_year_months"],
        ascending=False,
    )[cols].head(15)
)

Top by n_year_months


,h3_cell,pathogen_name,n_year_months,n_years,year_span,n_records,year_min,year_max
0,8429a87ffffffff,West Nile virus,96,16,22,756,2004,2025
1,84a81c7ffffffff,Dengue virus,72,13,16,1166,2010,2025
2,842b887ffffffff,West Nile virus,72,20,24,1111,2000,2023
3,8444a99ffffffff,Dengue virus,52,10,17,1272,2010,2026
4,848bb13ffffffff,Dengue virus,52,10,13,351,2012,2024
5,842f5bdffffffff,Dengue virus,51,10,14,110,2010,2023
6,846608dffffffff,Dengue virus,51,16,19,83,1998,2016
7,8460a65ffffffff,Dengue virus,50,12,13,1049,2012,2024
8,848db59ffffffff,Dengue virus,50,12,17,147,2010,2026
9,8441185ffffffff,Dengue virus,48,9,14,246,2012,2025


Top by n_years


,h3_cell,pathogen_name,n_year_months,n_years,year_span,n_records,year_min,year_max
34,842149bffffffff,Tick-borne encephalitis virus,27,21,31,57,1981,2011
2,842b887ffffffff,West Nile virus,72,20,24,1111,2000,2023
52,8440441ffffffff,Japanese encephalitis virus,20,16,48,56,1977,2024
0,8429a87ffffffff,West Nile virus,96,16,22,756,2004,2025
6,846608dffffffff,Dengue virus,51,16,19,83,1998,2016
54,8494511ffffffff,Bluetongue virus,19,15,35,194,1981,2015
39,84a7b4bffffffff,Sindbis virus,25,15,28,36,1987,2014
73,8430dabffffffff,Japanese encephalitis virus,17,14,21,48,2001,2021
10,8464b67ffffffff,Dengue virus,47,14,17,93,2007,2023
45,842e617ffffffff,Japanese encephalitis virus,22,14,15,106,1979,1993
